In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython())
print(f'IN_COLAB={IN_COLAB}')


### Environment detection
This first cell sets a shared environment flag so the notebook can run in Colab with Drive paths or locally with repository-relative paths.

### Setup paths and imports
We mount Drive on Colab, define stable training output paths, and add the project root to `sys.path` for clean package imports.

In [ ]:
import sys
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/piano_model')
    BASE_PATH.mkdir(parents=True, exist_ok=True)
    PROJECT_ROOT = BASE_PATH / 'piano_midi_model'
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
    BASE_PATH = PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT={PROJECT_ROOT}')


### Install dependencies
This installs Colab-specific dependencies (including CUDA Mamba package) in Colab and local dependencies otherwise.

In [ ]:
import subprocess
import sys

req_file = PROJECT_ROOT / ('requirements_colab.txt' if IN_COLAB else 'requirements.txt')
if req_file.exists():
    print(f'Installing from {req_file}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)], check=False)
else:
    print(f'Requirements file not found: {req_file}')


### Load configs and instantiate baseline model
We build the GRU baseline first as a sanity test for data loading, optimization, checkpointing, and generation.

In [ ]:
from config import DataConfig, ModelConfig, TrainConfig
from model.baseline import PianoBaselineModel

data_cfg = DataConfig()
train_cfg = TrainConfig(max_epochs=5, batch_size=8, grad_accumulation_steps=4, device='auto')
model_cfg = ModelConfig(use_mamba=False, use_cfc=False)

if IN_COLAB:
    data_cfg.maestro_path = str(BASE_PATH / 'maestro-v3.0.0')
    data_cfg.tokenizer_path = str(BASE_PATH / 'tokenizer.json')
    data_cfg.processed_path = str(BASE_PATH / 'processed')
    train_cfg.checkpoint_dir = str(BASE_PATH / 'checkpoints_baseline')

baseline_model = PianoBaselineModel(
    vocab_size=data_cfg.vocab_size,
    d_model=model_cfg.d_model,
    n_layers=2,
    dropout=model_cfg.dropout,
    max_sequence_length=data_cfg.max_sequence_length,
)
baseline_model


### Log model summary
This reports parameter counts and backend status so each experiment run starts with explicit architecture metadata.

In [ ]:
from utils.logging_utils import log_model_summary

log_model_summary(baseline_model, model_cfg)


### Create DataLoaders
We load the train/validation/test splits from `manifest.json`, with split-by-piece semantics to avoid leakage.

In [ ]:
from data.dataset import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(data_cfg, train_cfg)
print('Train batches:', len(train_loader))
print('Val batches:', len(val_loader))
print('Test batches:', len(test_loader))


### Instantiate trainer
The trainer configures optimizer, warmup+cosine LR schedule, mixed precision on CUDA, and checkpoint handling.

In [ ]:
from data.tokenizer import PianoTokenizer
from training.trainer import Trainer

tokenizer = PianoTokenizer.load(data_cfg.tokenizer_path)
trainer = Trainer(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_cfg,
    data_config=data_cfg,
    tokenizer=tokenizer,
)
trainer


### Train baseline for 5 epochs
This is a quick sanity run to verify end-to-end stability before introducing Mamba and CfC complexity.

In [ ]:
history = trainer.train()
history.keys()


### Plot train/validation loss curves
Loss curves provide immediate feedback on optimization behavior and overfitting during the sanity run.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='train_loss')
plt.plot(history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Baseline Training Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Generate one continuation, visualize, and render audio
We decode one test seed and its generated continuation for quick qualitative inspection with both piano roll and playable waveform.

In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

from IPython.display import Audio, display

from utils.midi_utils import compare_pianorolls

seed_batch, _ = next(iter(test_loader))
seed_tokens = seed_batch[0].tolist()
generated = baseline_model.generate(seed_tokens, max_new_tokens=data_cfg.continuation_length)
generated_continuation = generated[len(seed_tokens):]

out_dir = Path(train_cfg.checkpoint_dir) / 'qualitative'
out_dir.mkdir(parents=True, exist_ok=True)
seed_mid = out_dir / 'baseline_seed.mid'
gen_full_mid = out_dir / 'baseline_generated_full.mid'
gen_cont_mid = out_dir / 'baseline_generated_continuation.mid'
tokenizer.decode(seed_tokens, seed_mid)
tokenizer.decode(generated, gen_full_mid)
tokenizer.decode(generated_continuation, gen_cont_mid)

compare_pianorolls(seed_mid, gen_cont_mid)

wav_path = out_dir / 'baseline_generated.wav'
rendered = False
soundfont_candidates = [
    '/usr/share/sounds/sf2/FluidR3_GM.sf2',
    '/usr/share/soundfonts/default.sf2',
    '/usr/share/sounds/sf2/default.sf2',
]
soundfont = next((p for p in soundfont_candidates if Path(p).exists()), None)

if IN_COLAB and shutil.which('fluidsynth') is None:
    subprocess.run(['apt-get', '-y', 'install', 'fluidsynth'], check=False)

if shutil.which('fluidsynth') and soundfont:
    cmd = ['fluidsynth', '-ni', soundfont, str(gen_full_mid), '-F', str(wav_path), '-r', '44100']
    subprocess.run(cmd, check=False)
    rendered = wav_path.exists()
else:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'midi2audio'], check=False)
        from midi2audio import FluidSynth
        if soundfont:
            FluidSynth(sound_font=soundfont).midi_to_audio(str(gen_full_mid), str(wav_path))
            rendered = wav_path.exists()
    except Exception as exc:
        print(f'Audio rendering skipped: {exc}')

if rendered:
    display(Audio(filename=str(wav_path), rate=44100))
else:
    print('Could not render WAV. Install fluidsynth + a soundfont to enable audio playback.')


### Save an explicit final checkpoint
Even though periodic checkpoints are already saved, this writes a tagged final checkpoint for reproducible baseline comparison.

In [ ]:
final_val = history['val_loss'][-1] if history['val_loss'] else 0.0
trainer.save_checkpoint(epoch=train_cfg.max_epochs, val_loss=final_val, tag='baseline_final')
print(f'Checkpoint directory: {train_cfg.checkpoint_dir}')
